In [1]:
import fsspec
print(fsspec.__version__)
# leitura online, armazenando e metrica


2026.2.0


In [2]:
import os, warnings, io, re, unicodedata
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import requests

os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


In [3]:
from pandera.pandas import Column, Check, DataFrameSchema
import pandera
import pandera.pandas as pa

PAGINA_SINOPSE_2024 = 'https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/sinopses-estatisticas/educacao-basica'
URL_SINOPSE_2024 = 'https://download.inep.gov.br/dados_abertos/sinopses_estatisticas/sinopses_estatisticas_censo_escolar_2024.zip'
PAGINA_RENDIMENTO_2024 = 'https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais/taxas-de-rendimento-escolar/2024'
URL_RENDIMENTO_2024 = 'https://download.inep.gov.br/informacoes_estatisticas/indicadores_educacionais/2024/tx_rend_municipios_2024.zip'
N_VALIDACAO = 500_000
RANDOM_STATE = 42

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'dados').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

LOCAL_SINOPSE = PROJECT_ROOT / 'dados' / 'Sinopse_Estatistica_da_Educação_Basica_2024.xlsx'
LOCAL_RENDIMENTO = PROJECT_ROOT / 'dados' / 'tx_rend_municipios_2024.zip'

print(f'Projeto: {PROJECT_ROOT}')
print(f'Sinopse oficial: {URL_SINOPSE_2024}')
print(f'Rendimento oficial: {URL_RENDIMENTO_2024}')
print(f'Fallback local sinopse: {LOCAL_SINOPSE.exists()} -> {LOCAL_SINOPSE}')
print(f'Fallback local rendimento: {LOCAL_RENDIMENTO.exists()} -> {LOCAL_RENDIMENTO}')
print(f'pandas: {pd.__version__} | pandera: {pandera.__version__}')


Projeto: /home/raimundoivy/Documents/pad_avaliação_02
Sinopse oficial: https://download.inep.gov.br/dados_abertos/sinopses_estatisticas/sinopses_estatisticas_censo_escolar_2024.zip
Rendimento oficial: https://download.inep.gov.br/informacoes_estatisticas/indicadores_educacionais/2024/tx_rend_municipios_2024.zip
Fallback local sinopse: True -> /home/raimundoivy/Documents/pad_avaliação_02/dados/Sinopse_Estatistica_da_Educação_Basica_2024.xlsx
Fallback local rendimento: True -> /home/raimundoivy/Documents/pad_avaliação_02/dados/tx_rend_municipios_2024.zip
pandas: 3.0.1 | pandera: 0.30.1


In [4]:
# SINOPSE 2024 - metadados e recorte de matriculas rurais
print('Lendo metadados da sinopse...')

try:
    fs = fsspec.filesystem('https')
    info_sinopse = fs.info(URL_SINOPSE_2024)
    print('fsspec ok')
    print(f' Tamanho remoto: {info_sinopse.get("size", "n/d")}')
    print(f' Tipo remoto   : {info_sinopse.get("type", "n/d")}')
except Exception as e:
    print('fsspec https nao disponivel neste ambiente; seguindo com requests.head para os metadados')

r = requests.head(URL_SINOPSE_2024, allow_redirects=True, timeout=60, verify=False)
r.raise_for_status()
print(f'HTTP status    : {r.status_code}')
print(f'Content-Type   : {r.headers.get("Content-Type", "n/d")}')
print(f'Content-Length : {r.headers.get("Content-Length", "n/d")}')
print(f'Pagina oficial : {PAGINA_SINOPSE_2024}')

if LOCAL_SINOPSE.exists():
    fonte_sinopse = LOCAL_SINOPSE
    print(f'Usando fallback local: {fonte_sinopse.name}')
    sinopse_abas = pd.ExcelFile(fonte_sinopse).sheet_names
    frame_sinopse = pd.read_excel(fonte_sinopse, sheet_name='1.26', header=[5, 6, 7, 8])
else:
    print('Baixando zip remoto da sinopse...')
    r = requests.get(URL_SINOPSE_2024, timeout=180, verify=False)
    r.raise_for_status()
    with ZipFile(io.BytesIO(r.content)) as zf:
        zip_entries = zf.namelist()
        print('Entradas do zip (primeiras):')
        for item in zip_entries[:8]:
            print(' ', item)
        excel_name = next(name for name in zip_entries if name.endswith('.xlsx'))
        payload = io.BytesIO(zf.read(excel_name))
    sinopse_abas = pd.ExcelFile(payload).sheet_names
    payload.seek(0)
    frame_sinopse = pd.read_excel(payload, sheet_name='1.26', header=[5, 6, 7, 8])

print('Abas da sinopse (primeiras):')
for aba in sinopse_abas[:12]:
    print(' ', aba)
print(f'Dimensoes da aba 1.26: {frame_sinopse.shape}')

colunas_sinopse = []
for coluna in frame_sinopse.columns:
    if isinstance(coluna, tuple):
        codigo = str(coluna[-1]).strip()
        if codigo == 'Unnamed: 0_level_3':
            colunas_sinopse.append('regiao_inep')
        elif codigo == 'Unnamed: 1_level_3':
            colunas_sinopse.append('uf_extenso')
        elif codigo == 'Unnamed: 2_level_3':
            colunas_sinopse.append('nome_municipio_inep')
        elif codigo == 'Unnamed: 3_level_3':
            colunas_sinopse.append('codigo_municipio')
        else:
            partes = []
            for item in coluna:
                if pd.isna(item):
                    continue
                texto = unicodedata.normalize('NFKD', str(item))
                texto = texto.encode('ascii', 'ignore').decode('ascii')
                texto = ' '.join(texto.split()).strip()
                if texto:
                    partes.append(texto)
            colunas_sinopse.append(' | '.join(partes))
    else:
        colunas_sinopse.append(str(coluna))

frame_sinopse.columns = colunas_sinopse
coluna_rural_total = None
for coluna in frame_sinopse.columns:
    if str(coluna).startswith('Numero de Matriculas do Ensino Medio') and 'Rural' in str(coluna) and str(coluna).endswith('Total'):
        coluna_rural_total = coluna
        break

print(f'Coluna localizada: {coluna_rural_total}')

df_matriculas_raw = frame_sinopse[['codigo_municipio', coluna_rural_total]].copy()
df_matriculas_raw['codigo_municipio'] = (
    df_matriculas_raw['codigo_municipio']
    .astype('string')
    .str.strip()
    .str.replace('.0', '', regex=False)
    .str.extract(r'(\d+)', expand=False)
    .str.zfill(7)
)
df_matriculas_raw = df_matriculas_raw[df_matriculas_raw['codigo_municipio'].notna()].copy()
df_matriculas_raw[coluna_rural_total] = pd.to_numeric(df_matriculas_raw[coluna_rural_total], errors='coerce')
df_matriculas_raw = df_matriculas_raw.rename(columns={coluna_rural_total: 'matriculas_ensino_medio_rural_2024'})

print('=' * 70)
print('Recorte de matriculas rurais')
print('=' * 70)
print(f'Linhas            : {len(df_matriculas_raw):,}')
print(f'Municipios unicos : {df_matriculas_raw["codigo_municipio"].nunique():,}')
print(f'Nulos             : {df_matriculas_raw["matriculas_ensino_medio_rural_2024"].isna().sum():,}')
print(f'Positivos         : {df_matriculas_raw["matriculas_ensino_medio_rural_2024"].fillna(0).gt(0).sum():,}')
print()
print(df_matriculas_raw.head(10).to_string(index=False))
print()
print(df_matriculas_raw['matriculas_ensino_medio_rural_2024'].describe())


Lendo metadados da sinopse...
fsspec https nao disponivel neste ambiente; seguindo com requests.head para os metadados


HTTP status    : 200
Content-Type   : application/zip
Content-Length : 144409489
Pagina oficial : https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/sinopses-estatisticas/educacao-basica
Usando fallback local: Sinopse_Estatistica_da_Educação_Basica_2024.xlsx


Abas da sinopse (primeiras):
  Sinopse Estatística
  Resumo
  Equipe Técnica
  Como Citar
  Sumário
  1 - Matrículas
  Educação Básica 1.1
  1.2
  1.3
  1.4
  Educação Infantil 1.5
  Creche 1.6
Dimensoes da aba 1.26: (5610, 15)
Coluna localizada: Numero de Matriculas do Ensino Medio | Localizacao e Dependencia Administrativa | Rural | Total
Recorte de matriculas rurais
Linhas            : 5,570
Municipios unicos : 5,570
Nulos             : 0
Positivos         : 1,612

codigo_municipio  matriculas_ensino_medio_rural_2024
         1100015                                72.0
         1100379                                 0.0
         1100403                                 0.0
         1100346                                 0.0
         1100023                               703.0
         1100452                                 0.0
         1100031                                33.0
         1100601                                 0.0
         1100049                               189

In [5]:
# RENDIMENTO 2024 - metadados e leitura da planilha original
print('Lendo metadados do rendimento escolar...')

try:
    fs = fsspec.filesystem('https')
    info_rendimento = fs.info(URL_RENDIMENTO_2024)
    print('fsspec ok')
    print(f' Tamanho remoto: {info_rendimento.get("size", "n/d")}')
    print(f' Tipo remoto   : {info_rendimento.get("type", "n/d")}')
except Exception as e:
    print('fsspec https nao disponivel neste ambiente; seguindo com requests.head para os metadados')

r = requests.head(URL_RENDIMENTO_2024, allow_redirects=True, timeout=60, verify=False)
r.raise_for_status()
print(f'HTTP status    : {r.status_code}')
print(f'Content-Type   : {r.headers.get("Content-Type", "n/d")}')
print(f'Content-Length : {r.headers.get("Content-Length", "n/d")}')
print(f'Pagina oficial : {PAGINA_RENDIMENTO_2024}')

if LOCAL_RENDIMENTO.exists():
    fonte_rendimento = LOCAL_RENDIMENTO
    print(f'Usando fallback local: {fonte_rendimento.name}')
    with ZipFile(fonte_rendimento) as zf:
        zip_entries = zf.namelist()
        print('Entradas do zip (primeiras):')
        for item in zip_entries[:8]:
            print(' ', item)
        excel_name = next(name for name in zip_entries if name.endswith('.xlsx'))
        payload = io.BytesIO(zf.read(excel_name))
else:
    print('Baixando zip remoto do rendimento...')
    r = requests.get(URL_RENDIMENTO_2024, timeout=180, verify=False)
    r.raise_for_status()
    with ZipFile(io.BytesIO(r.content)) as zf:
        zip_entries = zf.namelist()
        print('Entradas do zip (primeiras):')
        for item in zip_entries[:8]:
            print(' ', item)
        excel_name = next(name for name in zip_entries if name.endswith('.xlsx'))
        payload = io.BytesIO(zf.read(excel_name))

rendimento_abas = pd.ExcelFile(payload).sheet_names
print('Abas do arquivo:')
for aba in rendimento_abas:
    print(' ', aba)

payload.seek(0)
frame_rendimento = pd.read_excel(payload, sheet_name='MUNICIPIOS ', header=[5, 6, 7, 8])
print(f'Dimensoes da planilha original: {frame_rendimento.shape}')


Lendo metadados do rendimento escolar...
fsspec https nao disponivel neste ambiente; seguindo com requests.head para os metadados


HTTP status    : 200
Content-Type   : application/zip
Content-Length : 34487202
Pagina oficial : https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais/taxas-de-rendimento-escolar/2024
Usando fallback local: tx_rend_municipios_2024.zip
Entradas do zip (primeiras):
  tx_rend_municipios_2024/
  tx_rend_municipios_2024/md5_tx_rend_municipios_2024.txt
  tx_rend_municipios_2024/tx_rend_municipios_2024.ods
  tx_rend_municipios_2024/tx_rend_municipios_2024.xlsx
Abas do arquivo:
  MUNICIPIOS 


Dimensoes da planilha original: (65595, 61)


In [6]:
# limpeza do cabecalho, mapeamento das medidas e transformacao para base longa
metadata_medidas = {}
colunas_flat = []

for coluna in frame_rendimento.columns:
    if isinstance(coluna, tuple):
        codigo = str(coluna[-1]).strip()
        colunas_flat.append(codigo)
        if re.match(r'^\d+_', codigo):
            partes = []
            for item in coluna[:3]:
                if pd.isna(item):
                    partes.append('')
                    continue
                texto = unicodedata.normalize('NFKD', str(item))
                texto = texto.encode('ascii', 'ignore').decode('ascii')
                texto = ' '.join(texto.split()).strip().lower()
                texto = re.sub(r'[^a-z0-9]+', '_', texto).strip('_')
                partes.append(texto)
            indicador, etapa_ensino, serie = partes
            if indicador and etapa_ensino and serie:
                metadata_medidas[codigo] = {
                    'indicador': indicador,
                    'etapa_ensino': etapa_ensino,
                    'serie': serie,
                }
    else:
        colunas_flat.append(str(coluna))

frame_rendimento.columns = colunas_flat
frame_rendimento = frame_rendimento.rename(columns={
    'CO_MUNICIPIO': 'codigo_municipio',
    'NO_CATEGORIA': 'localizacao',
    'NO_DEPENDENCIA': 'dependencia_administrativa',
})

frame_rendimento['codigo_municipio'] = (
    frame_rendimento['codigo_municipio']
    .astype('string')
    .str.strip()
    .str.replace('.0', '', regex=False)
    .str.extract(r'(\d+)', expand=False)
    .str.zfill(7)
)
frame_rendimento = frame_rendimento[frame_rendimento['codigo_municipio'].notna()].copy()
frame_rendimento['localizacao'] = frame_rendimento['localizacao'].astype('string').str.strip()
frame_rendimento['dependencia_administrativa'] = frame_rendimento['dependencia_administrativa'].astype('string').str.strip()

measure_columns = [col for col in frame_rendimento.columns if re.match(r'^\d+_', str(col))]
print(f'Colunas de medida encontradas: {len(measure_columns)}')
print('Primeiras medidas:')
for medida in measure_columns[:10]:
    print(' ', medida, '->', metadata_medidas.get(medida, {}))

print()
print('Convertendo para base longa...')

melted = frame_rendimento.melt(
    id_vars=['codigo_municipio', 'localizacao', 'dependencia_administrativa'],
    value_vars=measure_columns,
    var_name='medida',
    value_name='valor',
)

metadata_frame = pd.DataFrame.from_dict(metadata_medidas, orient='index').reset_index().rename(columns={'index': 'medida'})
df_rendimento_long = melted.merge(metadata_frame, on='medida', how='left')
df_rendimento_long['valor'] = pd.to_numeric(
    df_rendimento_long['valor'].astype('string').str.replace('--', '', regex=False).str.replace(',', '.', regex=False),
    errors='coerce'
)
df_rendimento_long = df_rendimento_long[df_rendimento_long['indicador'].notna()].reset_index(drop=True)

print(f'Linhas da base longa : {len(df_rendimento_long):,}')
print(f'Colunas da base longa: {df_rendimento_long.shape[1]}')
print(f'Municipios unicos    : {df_rendimento_long["codigo_municipio"].nunique():,}')
print(f'Indicadores distintos: {df_rendimento_long["indicador"].nunique():,}')
print(f'Etapas distintas     : {df_rendimento_long["etapa_ensino"].nunique():,}')
print(f'Series distintas     : {df_rendimento_long["serie"].nunique():,}')
print(f'Nulos em valor       : {df_rendimento_long["valor"].isna().sum():,}')
print()
print(df_rendimento_long.head(10).to_string(index=False))
print()
print('Tipos inferidos:')
print(df_rendimento_long.dtypes)
print()
print('Localizacoes observadas:')
for item in sorted(df_rendimento_long['localizacao'].dropna().astype(str).unique().tolist()):
    print(' ', item)
print()
print('Dependencias observadas:')
for item in sorted(df_rendimento_long['dependencia_administrativa'].dropna().astype(str).unique().tolist()):
    print(' ', item)
print()
print('Resumo numerico de valor:')
print(df_rendimento_long['valor'].describe())


Colunas de medida encontradas: 54
Primeiras medidas:
  1_CAT_FUN -> {'indicador': 'taxa_de_aprovacao', 'etapa_ensino': 'ensino_fundamental_de_8_e_9_anos', 'serie': 'total'}
  1_CAT_FUN_AI -> {'indicador': 'taxa_de_aprovacao', 'etapa_ensino': 'ensino_fundamental_de_8_e_9_anos', 'serie': 'anos_iniciais'}
  1_CAT_FUN_AF -> {'indicador': 'taxa_de_aprovacao', 'etapa_ensino': 'ensino_fundamental_de_8_e_9_anos', 'serie': 'anos_finais'}
  1_CAT_FUN_01 -> {'indicador': 'taxa_de_aprovacao', 'etapa_ensino': 'ensino_fundamental_de_8_e_9_anos', 'serie': '1o_ano'}
  1_CAT_FUN_02 -> {'indicador': 'taxa_de_aprovacao', 'etapa_ensino': 'ensino_fundamental_de_8_e_9_anos', 'serie': '2o_ano'}
  1_CAT_FUN_03 -> {'indicador': 'taxa_de_aprovacao', 'etapa_ensino': 'ensino_fundamental_de_8_e_9_anos', 'serie': '3o_ano'}
  1_CAT_FUN_04 -> {'indicador': 'taxa_de_aprovacao', 'etapa_ensino': 'ensino_fundamental_de_8_e_9_anos', 'serie': '4o_ano'}
  1_CAT_FUN_05 -> {'indicador': 'taxa_de_aprovacao', 'etapa_ensino': 'e

Linhas da base longa : 3,542,076
Colunas da base longa: 8
Municipios unicos    : 5,570
Indicadores distintos: 3
Etapas distintas     : 2
Series distintas     : 17
Nulos em valor       : 1,088,139

codigo_municipio localizacao dependencia_administrativa    medida  valor         indicador                     etapa_ensino serie
         1100015       Total                      Total 1_CAT_FUN   96.8 taxa_de_aprovacao ensino_fundamental_de_8_e_9_anos total
         1100015      Urbana                      Total 1_CAT_FUN   97.1 taxa_de_aprovacao ensino_fundamental_de_8_e_9_anos total
         1100015       Rural                      Total 1_CAT_FUN   96.3 taxa_de_aprovacao ensino_fundamental_de_8_e_9_anos total
         1100015       Total                   Estadual 1_CAT_FUN   97.1 taxa_de_aprovacao ensino_fundamental_de_8_e_9_anos total
         1100015      Urbana                   Estadual 1_CAT_FUN   97.4 taxa_de_aprovacao ensino_fundamental_de_8_e_9_anos total
         1100015       

count    2453937.0
mean     33.333333
std      44.131265
min            0.0
25%            0.0
50%            2.9
75%           93.0
max          100.0
Name: valor, dtype: Float64


In [7]:
# schema pandera e validacao da amostra
indicadores_validos = sorted(metadata_frame['indicador'].dropna().astype(str).unique().tolist())
medidas_validas = sorted(metadata_frame['medida'].dropna().astype(str).unique().tolist())
indicador_por_medida = dict(zip(metadata_frame['medida'], metadata_frame['indicador']))
etapa_por_medida = dict(zip(metadata_frame['medida'], metadata_frame['etapa_ensino']))
serie_por_medida = dict(zip(metadata_frame['medida'], metadata_frame['serie']))

dependencias_validas = ['Federal', 'Estadual', 'Municipal', 'Privada', 'Pública', 'Total']
localizacoes_validas = ['Rural', 'Urbana', 'Total']

schema_rendimento = DataFrameSchema(
    {
        'codigo_municipio': Column(str, Check.str_matches(r'^\d{7}$'), nullable=False),
        'localizacao': Column(str, Check.isin(localizacoes_validas), nullable=False),
        'dependencia_administrativa': Column(str, Check.isin(dependencias_validas), nullable=False),
        'medida': Column(str, Check.isin(medidas_validas), nullable=False),
        'indicador': Column(str, [Check.str_length(min_value=1), Check.isin(indicadores_validos)], nullable=False),
        'etapa_ensino': Column(str, Check.str_length(min_value=1), nullable=False),
        'serie': Column(str, Check.str_length(min_value=1), nullable=False),
        'valor': Column(float, nullable=True),
    },
    checks=[
        Check(lambda df: df['valor'].dropna().ge(0).all(), error='valor_nao_pode_ser_negativo'),
        Check(lambda df: df.loc[df['indicador'].eq('taxa_de_abandono'), 'valor'].dropna().between(0, 100).all(), error='taxa_de_abandono_entre_0_e_100'),
        Check(lambda df: df['medida'].map(indicador_por_medida).eq(df['indicador']).all(), error='indicador_incompativel_com_medida'),
        Check(lambda df: df['medida'].map(etapa_por_medida).eq(df['etapa_ensino']).all(), error='etapa_incompativel_com_medida'),
        Check(lambda df: df['medida'].map(serie_por_medida).eq(df['serie']).all(), error='serie_incompativel_com_medida'),
    ],
    coerce=True,
    strict=True,
    name='rendimento_escolar_2024',
)
print('Schema definido')
print(f'Colunas validadas: {len(schema_rendimento.columns)}')

if len(df_rendimento_long) < N_VALIDACAO:
    raise ValueError(f'A base longa tem {len(df_rendimento_long):,} linhas e nao atende ao minimo de {N_VALIDACAO:,}.')

print(f'Carregando amostra de {N_VALIDACAO:,} linhas ...')
sample_validacao = df_rendimento_long.sample(n=N_VALIDACAO, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Amostra pronta: {len(sample_validacao):,} linhas')
print(f'Memoria: {sample_validacao.memory_usage(deep=True).sum()/1e6:.1f} MB')
print()
print(sample_validacao.head(10).to_string(index=False))
print()

relatorio_falhas = pd.DataFrame()
resumo_falhas = pd.DataFrame()
status_validacao = 'ok'

try:
    schema_rendimento.validate(sample_validacao, lazy=True)
    print(f'Amostra valida ({len(sample_validacao):,} linhas)')
    resumo_falhas = pd.DataFrame([
        {'schema': 'rendimento_amostra', 'status': 'ok', 'linhas_validadas': len(sample_validacao), 'falhas': 0}
    ])
    print(resumo_falhas.to_string(index=False))
except pa.errors.SchemaErrors as e:
    status_validacao = 'falhas_encontradas'
    relatorio_falhas = e.failure_cases[['column', 'check', 'failure_case']].copy()
    print(f'Falhas detectadas: {len(relatorio_falhas):,} em {len(sample_validacao):,} linhas')
    print(f'Taxa: {len(relatorio_falhas)/len(sample_validacao)*100:.3f}%')
    print()

    resumo_falhas = (
        relatorio_falhas
        .groupby(['column', 'check'])['failure_case']
        .agg(n_falhas='count', exemplos=lambda x: list(pd.Series(x).dropna().astype(str).unique()[:3]))
        .reset_index()
        .sort_values('n_falhas', ascending=False)
    )
    print(resumo_falhas.to_string(index=False))


Schema definido
Colunas validadas: 8
Carregando amostra de 500,000 linhas ...


Amostra pronta: 500,000 linhas
Memoria: 73.5 MB

codigo_municipio localizacao dependencia_administrativa       medida  valor          indicador                     etapa_ensino    serie
         2609154       Total                      Total 3_CAT_FUN_07    3.2   taxa_de_abandono ensino_fundamental_de_8_e_9_anos   7o_ano
         3125903       Total                   Estadual 1_CAT_FUN_07   95.8  taxa_de_aprovacao ensino_fundamental_de_8_e_9_anos   7o_ano
         5215231       Rural                    Pública 2_CAT_FUN_03   50.0 taxa_de_reprovacao ensino_fundamental_de_8_e_9_anos   3o_ano
         4110078      Urbana                      Total 3_CAT_FUN_04    0.0   taxa_de_abandono ensino_fundamental_de_8_e_9_anos   4o_ano
         3168507       Total                      Total 2_CAT_FUN_05    0.0 taxa_de_reprovacao ensino_fundamental_de_8_e_9_anos   5o_ano
         2401909      Urbana                      Total 2_CAT_FUN_07    8.2 taxa_de_reprovacao ensino_fundamental_de_8_e_9_anos  

Amostra valida (500,000 linhas)
            schema status  linhas_validadas  falhas
rendimento_amostra     ok            500000       0


In [8]:
print('=' * 70)
print('Sintese final')
print('=' * 70)
print(f'Fonte oficial 1 : {PAGINA_SINOPSE_2024}')
print(f'Fonte oficial 2 : {PAGINA_RENDIMENTO_2024}')
print('DataFrame validado: df_rendimento_long')
print(f'Linhas totais     : {len(df_rendimento_long):,}')
print(f'Linhas na amostra : {len(sample_validacao):,}')
print(f'Resultado final   : {status_validacao}')


Sintese final
Fonte oficial 1 : https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/sinopses-estatisticas/educacao-basica
Fonte oficial 2 : https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais/taxas-de-rendimento-escolar/2024
DataFrame validado: df_rendimento_long
Linhas totais     : 3,542,076
Linhas na amostra : 500,000
Resultado final   : ok
